# Notebook 3: Mistral API — La Plateforme

**Goal:** Get fluent with the `mistralai` Python SDK — this is Mistral's own platform and they'll expect you to use it live.

**Topics:**
1. Setup & authentication
2. Chat completions
3. Streaming
4. Tool/Function calling
5. Embeddings
6. Structured output with Pydantic
7. Async calls
8. Retry logic & error handling
9. Token counting & cost estimation
10. JSON mode

> Get your free API key at: https://console.mistral.ai/
> `pip install mistralai` to install the SDK

In [ ]:
# Install if needed
# !pip install mistralai pydantic

import os
import json
import time
import asyncio
from typing import Optional, List, Dict, Any
from pydantic import BaseModel, Field

# Set your API key
# Option 1: Set environment variable: export MISTRAL_API_KEY="your-key"
# Option 2: Put it here (don't commit!)
MISTRAL_API_KEY = os.getenv('MISTRAL_API_KEY', 'YOUR_KEY_HERE')

print('API key set:', MISTRAL_API_KEY[:8] + '...' if len(MISTRAL_API_KEY) > 8 else 'NOT SET')

In [ ]:
try:
    from mistralai import Mistral
    from mistralai.models import ChatMessage
    print('mistralai SDK loaded successfully')
    MISTRAL_AVAILABLE = True
except ImportError:
    print('Run: pip install mistralai')
    MISTRAL_AVAILABLE = False

---
## 1. Basic Chat Completion

**Mistral model IDs:**
- `mistral-small-latest` — fast, cheap
- `mistral-medium-latest` — balanced
- `mistral-large-latest` — most capable
- `open-mistral-7b` — open source 7B
- `open-mixtral-8x7b` — open source MoE
- `codestral-latest` — code specialized

In [ ]:
def simple_chat(prompt: str, model: str = 'mistral-small-latest') -> str:
    """
    Send a single message and return the text response.
    """
    # YOUR CODE HERE
    # 1. Create Mistral client
    # 2. Call client.chat.complete()
    # 3. Return response.choices[0].message.content
    raise NotImplementedError()


# response = simple_chat('What is the difference between top-k and top-p sampling?')
# print(response)

In [ ]:
# REFERENCE
if MISTRAL_AVAILABLE:
    client = Mistral(api_key=MISTRAL_API_KEY)

    def simple_chat_ref(prompt: str, model: str = 'mistral-small-latest') -> str:
        response = client.chat.complete(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return response.choices[0].message.content
    
    simple_chat = simple_chat_ref
    
    # Test (will only run with valid API key)
    if MISTRAL_API_KEY != 'YOUR_KEY_HERE':
        response = simple_chat('In one sentence, what is the difference between top-k and top-p sampling?')
        print('Response:', response)
    else:
        print('[Skipped: no API key set]')

---
## 2. Multi-Turn Conversation

Maintain message history and pass it back each time.

In [ ]:
class ChatSession:
    """
    A stateful chat session that maintains conversation history.
    """
    def __init__(self, system_prompt: str, model: str = 'mistral-small-latest'):
        self.model = model
        self.messages: List[Dict[str, str]] = [
            {'role': 'system', 'content': system_prompt}
        ]
    
    def chat(self, user_message: str) -> str:
        """
        Add user message, call API, add assistant response, return response text.
        """
        # YOUR CODE HERE
        # 1. Append user message to self.messages
        # 2. Call API with full self.messages
        # 3. Extract assistant response
        # 4. Append assistant response to self.messages
        # 5. Return response text
        raise NotImplementedError()


# session = ChatSession('You are a helpful AI tutor teaching LLM concepts.')
# print(session.chat('What is attention?'))
# print(session.chat('How is it used in Mistral specifically?'))

In [ ]:
# REFERENCE
if MISTRAL_AVAILABLE:
    class ChatSessionRef:
        def __init__(self, system_prompt: str, model: str = 'mistral-small-latest'):
            self.model = model
            self.messages: List[Dict[str, str]] = [
                {'role': 'system', 'content': system_prompt}
            ]
        
        def chat(self, user_message: str) -> str:
            self.messages.append({'role': 'user', 'content': user_message})
            
            response = client.chat.complete(
                model=self.model,
                messages=self.messages,
            )
            
            assistant_msg = response.choices[0].message.content
            self.messages.append({'role': 'assistant', 'content': assistant_msg})
            return assistant_msg
    
    ChatSession = ChatSessionRef
    
    if MISTRAL_API_KEY != 'YOUR_KEY_HERE':
        session = ChatSession('You are a concise AI assistant.')
        r1 = session.chat('What is KV caching?')
        print('Turn 1:', r1[:200])
        r2 = session.chat('What does it save exactly?')
        print('Turn 2:', r2[:200])
        print('\nTotal messages in history:', len(session.messages))
    else:
        print('[Skipped: no API key]')

---
## 3. Streaming Responses

For production apps, streaming lets you show tokens as they're generated — much better UX.

In [ ]:
def stream_chat(prompt: str, model: str = 'mistral-small-latest') -> str:
    """
    Stream a chat response, printing tokens as they arrive.
    Returns the full accumulated text.
    """
    # YOUR CODE HERE
    # Use client.chat.stream() context manager
    # Iterate over events
    # Each event has: event.data.choices[0].delta.content
    raise NotImplementedError()

# stream_chat('Count from 1 to 5 slowly.')

In [ ]:
# REFERENCE
if MISTRAL_AVAILABLE:
    def stream_chat_ref(prompt: str, model: str = 'mistral-small-latest') -> str:
        full_text = ''
        
        with client.chat.stream(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        ) as stream:
            for event in stream:
                chunk = event.data.choices[0].delta.content
                if chunk:
                    print(chunk, end='', flush=True)
                    full_text += chunk
        
        print()  # newline at end
        return full_text
    
    stream_chat = stream_chat_ref
    
    if MISTRAL_API_KEY != 'YOUR_KEY_HERE':
        result = stream_chat('Name 3 key differences between Mistral 7B and Mixtral 8x7B.')
    else:
        print('[Skipped: no API key]')

---
## 4. Tool / Function Calling

Mistral supports tool calling — the model decides when to call a function and with what arguments.

**Flow:**
1. Define tools with JSON schema
2. Send to model → it may return a tool call instead of text
3. Execute the tool in your code
4. Send tool result back to model
5. Model generates final response

In [ ]:
# Define tools as JSON schema
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_weather',
            'description': 'Get current weather for a city.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'city': {
                        'type': 'string',
                        'description': 'City name, e.g. Paris'
                    },
                    'unit': {
                        'type': 'string',
                        'enum': ['celsius', 'fahrenheit'],
                        'description': 'Temperature unit'
                    }
                },
                'required': ['city']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'calculate',
            'description': 'Perform a mathematical calculation.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'expression': {
                        'type': 'string',
                        'description': 'Math expression to evaluate, e.g. "2 + 2"'
                    }
                },
                'required': ['expression']
            }
        }
    }
]

# Mock tool executors
def get_weather(city: str, unit: str = 'celsius') -> str:
    return json.dumps({'city': city, 'temperature': 22, 'unit': unit, 'condition': 'sunny'})

def calculate(expression: str) -> str:
    try:
        result = eval(expression, {'__builtins__': {}}, {})
        return json.dumps({'result': result})
    except Exception as e:
        return json.dumps({'error': str(e)})

TOOL_MAP = {'get_weather': get_weather, 'calculate': calculate}

print('Tools defined:', [t['function']['name'] for t in tools])

In [ ]:
def tool_calling_agent(user_message: str, model: str = 'mistral-small-latest') -> str:
    """
    Agent that can use tools to answer questions.
    Implements the full tool-calling loop.
    """
    # YOUR CODE HERE
    # 1. Send user message + tools to model
    # 2. Check if response has tool_calls
    # 3. If yes: execute each tool call, append results
    # 4. Send tool results back to model
    # 5. Return final text response
    raise NotImplementedError()

# result = tool_calling_agent('What is the weather in Paris and what is 42 * 17?')
# print(result)

In [ ]:
# REFERENCE
if MISTRAL_AVAILABLE:
    def tool_calling_agent_ref(user_message: str, model: str = 'mistral-small-latest') -> str:
        messages = [{'role': 'user', 'content': user_message}]
        
        # First call: model may return tool calls
        response = client.chat.complete(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice='auto',
        )
        
        msg = response.choices[0].message
        
        # If no tool calls, return text directly
        if not msg.tool_calls:
            return msg.content
        
        # Append assistant message with tool calls
        messages.append({'role': 'assistant', 'content': msg.content, 'tool_calls': msg.tool_calls})
        
        # Execute each tool call
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            
            print(f'Calling tool: {fn_name}({fn_args})')
            result = TOOL_MAP[fn_name](**fn_args)
            
            messages.append({
                'role': 'tool',
                'name': fn_name,
                'content': result,
                'tool_call_id': tool_call.id,
            })
        
        # Second call: get final answer with tool results
        final_response = client.chat.complete(
            model=model,
            messages=messages,
            tools=tools,
        )
        
        return final_response.choices[0].message.content
    
    tool_calling_agent = tool_calling_agent_ref
    
    if MISTRAL_API_KEY != 'YOUR_KEY_HERE':
        result = tool_calling_agent('What is the weather in Paris and what is 42 * 17?')
        print('\nFinal answer:', result)
    else:
        print('[Skipped: no API key]')

---
## 5. Embeddings API

In [ ]:
import numpy as np

def get_embeddings(texts: List[str], model: str = 'mistral-embed') -> np.ndarray:
    """
    Get embeddings for a list of texts.
    Returns: numpy array of shape (len(texts), embedding_dim)
    """
    # YOUR CODE HERE
    # Use client.embeddings.create()
    # response.data is a list of embedding objects
    raise NotImplementedError()


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    # YOUR CODE HERE
    raise NotImplementedError()


# texts = [
#     'The transformer architecture uses self-attention.',
#     'Neural networks can learn patterns from data.',
#     'Paris is the capital of France.',
# ]
# embeddings = get_embeddings(texts)
# print('Embedding shape:', embeddings.shape)
# 
# # Semantic similarity
# sim_01 = cosine_similarity(embeddings[0], embeddings[1])
# sim_02 = cosine_similarity(embeddings[0], embeddings[2])
# print(f'Similarity (AI vs AI): {sim_01:.3f}')  # should be high
# print(f'Similarity (AI vs Paris): {sim_02:.3f}')  # should be lower

In [ ]:
# REFERENCE
if MISTRAL_AVAILABLE:
    def get_embeddings_ref(texts: List[str], model: str = 'mistral-embed') -> np.ndarray:
        response = client.embeddings.create(
            model=model,
            inputs=texts,
        )
        return np.array([item.embedding for item in response.data])
    
    def cosine_similarity_ref(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
    
    get_embeddings = get_embeddings_ref
    cosine_similarity = cosine_similarity_ref
    
    if MISTRAL_API_KEY != 'YOUR_KEY_HERE':
        texts = [
            'The transformer architecture uses self-attention.',
            'Neural networks can learn patterns from data.',
            'Paris is the capital of France.',
        ]
        embeddings = get_embeddings(texts)
        print('Embedding shape:', embeddings.shape)
        
        sim_01 = cosine_similarity(embeddings[0], embeddings[1])
        sim_02 = cosine_similarity(embeddings[0], embeddings[2])
        print(f'Similarity (AI topic vs AI topic): {sim_01:.3f}')
        print(f'Similarity (AI topic vs Paris): {sim_02:.3f}')
    else:
        print('[Skipped: no API key]')

---
## 6. Structured Output with Pydantic

Use JSON mode to get structured, typed responses. In production, parse with Pydantic.

**Interview Q:** *How would you ensure the model always returns valid JSON with a specific schema?*

In [ ]:
from pydantic import BaseModel, Field
from typing import List as TList

class ModelInfo(BaseModel):
    name: str = Field(description='Model name')
    parameter_count: str = Field(description='Number of parameters e.g. 7B')
    architecture: str = Field(description='Architecture type e.g. Dense, MoE')
    key_features: TList[str] = Field(description='List of key features')


def get_structured_model_info(model_name: str) -> ModelInfo:
    """
    Ask Mistral to describe an LLM model and parse the JSON response.
    """
    # YOUR CODE HERE
    # 1. Build prompt asking for JSON with ModelInfo schema
    # 2. Call with response_format={'type': 'json_object'}
    # 3. Parse with ModelInfo.model_validate_json(content)
    raise NotImplementedError()


# info = get_structured_model_info('Mixtral 8x7B')
# print(f'Name: {info.name}')
# print(f'Params: {info.parameter_count}')
# print(f'Architecture: {info.architecture}')
# print(f'Features: {info.key_features}')

In [ ]:
# REFERENCE
if MISTRAL_AVAILABLE:
    def get_structured_model_info_ref(model_name: str) -> ModelInfo:
        schema = ModelInfo.model_json_schema()
        prompt = f"""Provide information about {model_name} in JSON format matching this schema:
{json.dumps(schema, indent=2)}

Return only valid JSON, no markdown."""
        
        response = client.chat.complete(
            model='mistral-small-latest',
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object'},
        )
        
        content = response.choices[0].message.content
        return ModelInfo.model_validate_json(content)
    
    get_structured_model_info = get_structured_model_info_ref
    
    if MISTRAL_API_KEY != 'YOUR_KEY_HERE':
        info = get_structured_model_info('Mixtral 8x7B')
        print(f'Name: {info.name}')
        print(f'Params: {info.parameter_count}')
        print(f'Architecture: {info.architecture}')
        print(f'Features:')
        for f in info.key_features:
            print(f'  - {f}')
    else:
        print('[Skipped: no API key]')

---
## 7. Async API Calls

For high-throughput applications, use async to make many calls in parallel.

In [ ]:
async def async_chat(prompt: str, model: str = 'mistral-small-latest') -> str:
    """
    Async version of chat completion.
    """
    # YOUR CODE HERE
    # Use Mistral(api_key=...) async client
    # Or use AsyncMistral if available
    raise NotImplementedError()


async def parallel_chat(prompts: List[str]) -> List[str]:
    """
    Send multiple prompts in parallel using asyncio.gather().
    """
    # YOUR CODE HERE
    raise NotImplementedError()

# import asyncio
# prompts = ['What is top-k sampling?', 'What is RAG?', 'What is LoRA?']
# results = asyncio.run(parallel_chat(prompts))
# for p, r in zip(prompts, results):
#     print(f'Q: {p}\nA: {r[:100]}\n')

In [ ]:
# REFERENCE
if MISTRAL_AVAILABLE:
    from mistralai import AsyncMistral
    
    async_client = AsyncMistral(api_key=MISTRAL_API_KEY)
    
    async def async_chat_ref(prompt: str, model: str = 'mistral-small-latest') -> str:
        response = await async_client.chat.complete_async(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return response.choices[0].message.content
    
    async def parallel_chat_ref(prompts: List[str]) -> List[str]:
        tasks = [async_chat_ref(p) for p in prompts]
        return await asyncio.gather(*tasks)
    
    async_chat = async_chat_ref
    parallel_chat = parallel_chat_ref
    
    if MISTRAL_API_KEY != 'YOUR_KEY_HERE':
        prompts = [
            'In 10 words: what is top-k sampling?',
            'In 10 words: what is RAG?',
            'In 10 words: what is LoRA?',
        ]
        start = time.time()
        results = await parallel_chat(prompts)  # In Jupyter, can use await directly
        elapsed = time.time() - start
        print(f'3 parallel calls completed in {elapsed:.2f}s')
        for p, r in zip(prompts, results):
            print(f'\nQ: {p}\nA: {r}')
    else:
        print('[Skipped: no API key]')

---
## 8. Retry Logic with Exponential Backoff

Production code must handle rate limits and transient errors gracefully.

In [ ]:
import time
import random

def chat_with_retry(
    prompt: str,
    model: str = 'mistral-small-latest',
    max_retries: int = 5,
    base_delay: float = 1.0,
) -> str:
    """
    Chat with exponential backoff retry on rate limit errors.
    
    Retry on: RateLimitError, ServiceUnavailableError
    Do NOT retry on: AuthenticationError, InvalidRequestError
    """
    # YOUR CODE HERE
    # Use exponential backoff: delay = base_delay * 2^attempt + jitter
    # Jitter prevents thundering herd
    raise NotImplementedError()


# REFERENCE
if MISTRAL_AVAILABLE:
    from mistralai.models import SDKError
    
    def chat_with_retry_ref(
        prompt: str,
        model: str = 'mistral-small-latest',
        max_retries: int = 5,
        base_delay: float = 1.0,
    ) -> str:
        for attempt in range(max_retries):
            try:
                response = client.chat.complete(
                    model=model,
                    messages=[{'role': 'user', 'content': prompt}],
                )
                return response.choices[0].message.content
            
            except SDKError as e:
                if e.status_code in (429, 503) and attempt < max_retries - 1:
                    # Rate limit or service unavailable — retryable
                    delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
                    print(f'Retrying in {delay:.1f}s (attempt {attempt+1}/{max_retries})')
                    time.sleep(delay)
                else:
                    raise
        
        raise RuntimeError('Max retries exceeded')
    
    chat_with_retry = chat_with_retry_ref
    print('chat_with_retry implemented')

---
## 9. Token Counting & Cost Estimation

In [ ]:
# Approximate cost estimation
# Mistral pricing (check current at docs.mistral.ai/platform/pricing/)
PRICING = {
    'mistral-small-latest': {'input': 0.2, 'output': 0.6},   # $/M tokens
    'mistral-medium-latest': {'input': 2.7, 'output': 8.1},
    'mistral-large-latest': {'input': 2.0, 'output': 6.0},
    'mistral-embed': {'input': 0.1, 'output': 0.0},
}

def estimate_cost(
    input_tokens: int,
    output_tokens: int,
    model: str = 'mistral-small-latest'
) -> Dict[str, float]:
    """
    Estimate API cost given token counts.
    Returns dict with input_cost, output_cost, total_cost (all in USD).
    """
    # YOUR CODE HERE
    raise NotImplementedError()


# REFERENCE
def estimate_cost_ref(input_tokens: int, output_tokens: int, model: str = 'mistral-small-latest'):
    if model not in PRICING:
        raise ValueError(f'Unknown model: {model}')
    
    prices = PRICING[model]
    input_cost = (input_tokens / 1_000_000) * prices['input']
    output_cost = (output_tokens / 1_000_000) * prices['output']
    
    return {
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'input_cost': round(input_cost, 6),
        'output_cost': round(output_cost, 6),
        'total_cost': round(input_cost + output_cost, 6),
    }

estimate_cost = estimate_cost_ref

# Example: 1M input tokens, 100k output tokens
for model in PRICING:
    cost = estimate_cost(1_000_000, 100_000, model)
    print(f'{model}: ${cost["total_cost"]:.2f}')

---
## 10. Practice Exercise: Build a Mini Q&A Bot

Put it all together: a bot with system prompt, conversation history, tool support, and retry logic.

In [ ]:
class MistralQABot:
    """
    A production-quality Q&A bot using Mistral API.
    Features:
    - Stateful conversation
    - Streaming option
    - Cost tracking
    - Pydantic input validation
    """
    
    def __init__(
        self,
        system_prompt: str = 'You are a helpful AI assistant.',
        model: str = 'mistral-small-latest',
        max_tokens: Optional[int] = None,
        temperature: float = 0.7,
    ):
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def ask(self, question: str, stream: bool = False) -> str:
        """Ask a question, optionally stream the response."""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def reset(self):
        """Clear conversation history."""
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def get_usage_stats(self) -> Dict[str, Any]:
        """Return total tokens used and estimated cost."""
        # YOUR CODE HERE
        raise NotImplementedError()

# When implemented:
# bot = MistralQABot(system_prompt='You are an LLM expert. Be concise.')
# print(bot.ask('What is sliding window attention?'))
# print(bot.ask('How does that compare to full attention?'))
# print(bot.get_usage_stats())

---
## Interview Questions to Practice Answering

1. **API design:** If you were designing La Plateforme's rate limiting, how would you handle bursts? What data structure tracks per-user limits?

2. **Streaming:** What protocol does streaming use under the hood? (Server-Sent Events / HTTP chunked transfer)

3. **Tool calling:** How does the model 'know' to call a tool vs answer directly? (Training on tool-use data + system prompt)

4. **Cost optimization:** A customer is spending $10k/month on `mistral-large`. What would you recommend to reduce cost without sacrificing quality?

5. **JSON mode:** How do you guarantee valid JSON output? What if the model still produces invalid JSON?

---
**Next:** `04_rag_pipeline.ipynb`